# PySpark practice 15: production patterns

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-15-production-patterns.ipynb)

9 exercises on idempotent and incremental loads, CDC, SCD2, quality gates, testing and backfills. Each exercise names the [reference entries](pyspark-reference-15-production-patterns.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0 delta-spark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 15](pyspark-reference-15-production-patterns.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell. The session loads Delta Lake; `new_path(name)` returns a fresh table location.

In [ ]:
import os
import sys
import tempfile
import time

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the examples write

builder = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Delta's SQL: MERGE, time travel
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # fetches the Delta jars on the first run
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:110])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

import os
import uuid

from delta.tables import DeltaTable
from pyspark.testing.utils import assertSchemaEqual


def new_path(name):
    """A fresh table location under WORK_DIR."""
    return os.path.join(WORK_DIR, f"{name}-{uuid.uuid4().hex[:6]}")


def assert_df_equal(actual, expected):
    """Equal schemas and equal rows as multisets: the core of pyspark.testing.assertDataFrameEqual, without pandas."""
    assertSchemaEqual(actual.schema, expected.schema)
    missing, unexpected = expected.exceptAll(actual).count(), actual.exceptAll(expected).count()
    if missing or unexpected:
        raise AssertionError(f"DataFrames differ: {missing} expected row(s) missing, {unexpected} unexpected row(s)")

## Exercises

1 easy, 5 medium, 3 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Rerun-safe daily load](#exercise-1-rerun-safe-daily-load) | Medium | [Write patterns and idempotency](pyspark-reference-15-production-patterns.ipynb#write-patterns-and-idempotency) |
| 2 | [Incremental load with a high-water mark](#exercise-2-incremental-load-with-a-high-water-mark) | Hard | [Incremental loads with a high-water mark](pyspark-reference-15-production-patterns.ipynb#incremental-loads-with-a-high-water-mark) |
| 3 | [Apply a CDC batch with deletes](#exercise-3-apply-a-cdc-batch-with-deletes) | Hard | [Apply a CDC feed with inserts, updates and deletes](pyspark-reference-15-production-patterns.ipynb#apply-a-cdc-feed-with-inserts-updates-and-deletes) |
| 4 | [SCD type 2 update](#exercise-4-scd-type-2-update) | Hard | [SCD type 2 with one `MERGE`](pyspark-reference-15-production-patterns.ipynb#scd-type-2-with-one-merge) |
| 5 | [Quality gate with per-rule counts](#exercise-5-quality-gate-with-per-rule-counts) | Medium | [Quality gates and quarantine](pyspark-reference-15-production-patterns.ipynb#quality-gates-and-quarantine) |
| 6 | [Split valid rows from quarantine](#exercise-6-split-valid-rows-from-quarantine) | Medium | [Quality gates and quarantine](pyspark-reference-15-production-patterns.ipynb#quality-gates-and-quarantine) |
| 7 | [Unit-test a transformation](#exercise-7-unit-test-a-transformation) | Easy | [`assertDataFrameEqual` and `assertSchemaEqual`](pyspark-reference-15-production-patterns.ipynb#assertdataframeequal-and-assertschemaequal) |
| 8 | [Enforce a schema contract](#exercise-8-enforce-a-schema-contract) | Medium | [`assertDataFrameEqual` and `assertSchemaEqual`](pyspark-reference-15-production-patterns.ipynb#assertdataframeequal-and-assertschemaequal) |
| 9 | [Backfill a date range idempotently](#exercise-9-backfill-a-date-range-idempotently) | Medium | [Write patterns and idempotency](pyspark-reference-15-production-patterns.ipynb#write-patterns-and-idempotency) |

## Exercise 1: Rerun-safe daily load

**Difficulty:** Medium · about 15 min · **Practises:** [Write patterns and idempotency](pyspark-reference-15-production-patterns.ipynb#write-patterns-and-idempotency)

*Retries and backfills must not duplicate data.*

Write `load(run_date)` that recomputes order counts per status for one UTC day and replaces only that day in the Delta table at `daily` (partitioned by `day`). Load 2026-01-07, 2026-01-12 and 2026-01-07 again. Return `day`, `status`, `orders` ordered by `day`, `status`.

In [ ]:
daily = new_path("p15_daily")

**Expected output**

| day | status | orders |
| --- | --- | --- |
| 2026-01-07 | placed | 1 |
| 2026-01-07 | shipped | 1 |
| 2026-01-12 | paid | 1 |
| 2026-01-12 | shipped | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def load(run_date):
    rows = (orders.where(F.to_date("order_ts") == F.lit(run_date).cast("date"))
            .groupBy(F.to_date("order_ts").alias("day"), "status").agg(F.count("*").alias("orders")))
    if DeltaTable.isDeltaTable(spark, daily):
        rows.write.format("delta").mode("overwrite").option("replaceWhere", f"day = '{run_date}'").save(daily)
    else:
        rows.write.format("delta").partitionBy("day").save(daily)

for d in ["2026-01-07", "2026-01-12", "2026-01-07"]:
    load(d)
result = spark.read.format("delta").load(daily).orderBy("day", "status")
result.show()

## Exercise 2: Incremental load with a high-water mark

**Difficulty:** Hard · about 20 min · **Practises:** [Incremental loads with a high-water mark](pyspark-reference-15-production-patterns.ipynb#incremental-loads-with-a-high-water-mark)

*The standard pattern for loading only what changed.*

Implement `run(src)` that reads rows of `src` with `updated_at` above the mark in the control table, merges them into `silver` on `order_id`, then advances the mark, and returns the number of rows loaded. Run it on `v1`, `v2` and `v2` again; return `run` and `loaded`.

In [ ]:
silver, control = new_path("p15_silver"), new_path("p15_control")
spark.createDataFrame([(datetime(1970, 1, 1),)], "high_water timestamp").write.format("delta").save(control)
v1 = spark.createDataFrame([(1, "paid", datetime(2026, 1, 5)), (2, "placed", datetime(2026, 1, 6))], "order_id int, status string, updated_at timestamp")
v2 = v1.unionByName(spark.createDataFrame([(2, "shipped", datetime(2026, 1, 8)), (3, "paid", datetime(2026, 1, 8))], v1.schema))

**Expected output**

| run | loaded |
| --- | --- |
| 1 | 2 |
| 2 | 2 |
| 3 | 0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def run(src):
    mark = spark.read.format("delta").load(control).first().high_water
    new_rows = src.where(F.col("updated_at") > F.lit(mark))
    loaded = new_rows.count()
    if loaded == 0:
        return 0
    if DeltaTable.isDeltaTable(spark, silver):
        DeltaTable.forPath(spark, silver).alias("t").merge(new_rows.alias("s"), "t.order_id = s.order_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    else:
        new_rows.write.format("delta").save(silver)
    DeltaTable.forPath(spark, control).update(set={"high_water": F.lit(new_rows.agg(F.max("updated_at")).first()[0])})
    return loaded

result = spark.createDataFrame([(1, run(v1)), (2, run(v2)), (3, run(v2))], "run int, loaded int")
result.show()

## Exercise 3: Apply a CDC batch with deletes

**Difficulty:** Hard · about 20 min · **Practises:** [Apply a CDC feed with inserts, updates and deletes](pyspark-reference-15-production-patterns.ipynb#apply-a-cdc-feed-with-inserts-updates-and-deletes)

*CDC application is a core production pattern for lakehouses.*

Apply `cdc` (several changes per key; `op` is I/U/D; `lsn` orders them) to the Delta table at `tgt`: keep the latest change per key, delete on D, update on U, insert on I. Return the table ordered by `id`.

In [ ]:
tgt = new_path("p15_cdc")
spark.createDataFrame([(1, "a"), (2, "b"), (3, "c")], "id int, v string").write.format("delta").save(tgt)
cdc = spark.createDataFrame([(1, "a2", "U", 5), (1, "a3", "U", 6), (2, None, "D", 7), (4, "d", "I", 8), (3, "c2", "U", 9), (3, None, "D", 10)], "id int, v string, op string, lsn int")
cdc.show()

**Expected output**

| id | v |
| --- | --- |
| 1 | a3 |
| 4 | d |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
latest = cdc.withColumn("rn", F.row_number().over(Window.partitionBy("id").orderBy(F.desc("lsn")))).where("rn = 1")
(DeltaTable.forPath(spark, tgt).alias("t").merge(latest.alias("s"), "t.id = s.id")
 .whenMatchedDelete(condition="s.op = 'D'")
 .whenMatchedUpdate(condition="s.op <> 'D'", set={"v": "s.v"})
 .whenNotMatchedInsert(condition="s.op <> 'D'", values={"id": "s.id", "v": "s.v"})
 .execute())
result = spark.read.format("delta").load(tgt).orderBy("id")
result.show()

## Exercise 4: SCD type 2 update

**Difficulty:** Hard · about 25 min · **Practises:** [SCD type 2 with one `MERGE`](pyspark-reference-15-production-patterns.ipynb#scd-type-2-with-one-merge)

*Type 2 dimensions are asked about in nearly every data engineering interview.*

Apply `incoming` to the SCD2 dimension at `dim_path` at load time 2026-01-15: expire changed current rows (null-safe comparison of `region`) and insert new versions, and insert new customers. Return `customer_id`, `region`, `is_current`, ordered by `customer_id`, `is_current`.

In [ ]:
dim_path = new_path("p15_dim")
spark.createDataFrame([(1, "North", datetime(2024, 1, 1), None, True), (2, None, datetime(2024, 1, 1), None, True), (3, "South", datetime(2024, 1, 1), None, True)],
                      "customer_id int, region string, valid_from timestamp, valid_to timestamp, is_current boolean").write.format("delta").save(dim_path)
incoming = spark.createDataFrame([(1, "West"), (2, None), (3, "South"), (4, "East")], "customer_id int, region string")
load_ts = datetime(2026, 1, 15)

**Expected output**

| customer_id | region | is_current |
| --- | --- | --- |
| 1 | North | false |
| 1 | West | true |
| 2 | null | true |
| 3 | South | true |
| 4 | East | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
current = spark.read.format("delta").load(dim_path).where("is_current")
changed = incoming.alias("i").join(current.alias("c"), "customer_id").where(~F.col("i.region").eqNullSafe(F.col("c.region"))).select("i.*")
staged = changed.withColumn("merge_key", F.lit(None).cast("int")).unionByName(incoming.withColumn("merge_key", F.col("customer_id")))
(DeltaTable.forPath(spark, dim_path).alias("t")
 .merge(staged.alias("s"), "t.customer_id = s.merge_key AND t.is_current")
 .whenMatchedUpdate(condition="NOT (t.region <=> s.region)", set={"valid_to": F.lit(load_ts), "is_current": F.lit(False)})
 .whenNotMatchedInsert(values={"customer_id": "s.customer_id", "region": "s.region", "valid_from": F.lit(load_ts), "valid_to": F.lit(None).cast("timestamp"), "is_current": F.lit(True)})
 .execute())
result = spark.read.format("delta").load(dim_path).select("customer_id", "region", "is_current").orderBy("customer_id", "is_current")
result.show()

## Exercise 5: Quality gate with per-rule counts

**Difficulty:** Medium · about 15 min · **Practises:** [Quality gates and quarantine](pyspark-reference-15-production-patterns.ipynb#quality-gates-and-quarantine)

*Publish metrics per rule so failures are visible and trendable.*

Check `batch` against rules `null_customer` (customer_id null), `negative_amount` and `bad_status` (not in the four known statuses). Return `rule` and `failures` for each rule (in that order), plus a final row `rows_ok` with the number of rows that pass every rule.

In [ ]:
batch = orders.unionByName(spark.createDataFrame([(13, None, "paid", datetime(2026, 1, 14), D("10.00")), (14, 2, "lost", datetime(2026, 1, 14), D("-3.00"))], orders.schema))

**Expected output**

| rule | failures |
| --- | --- |
| null_customer | 1 |
| negative_amount | 1 |
| bad_status | 1 |
| rows_ok | 12 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
rules = {
    "null_customer": F.col("customer_id").isNull(),
    "negative_amount": F.col("amount") < 0,
    "bad_status": ~F.col("status").isin("placed", "paid", "shipped", "returned"),
}
counts = batch.agg(*[F.count_if(cond).alias(name) for name, cond in rules.items()]).first()
any_failure = F.lit(False)
for cond in rules.values():
    any_failure = any_failure | F.coalesce(cond, F.lit(False))
ok = batch.where(~any_failure).count()
result = spark.createDataFrame([(name, counts[name]) for name in rules] + [("rows_ok", ok)], "rule string, failures long")
result.show()

## Exercise 6: Split valid rows from quarantine

**Difficulty:** Medium · about 10 min · **Practises:** [Quality gates and quarantine](pyspark-reference-15-production-patterns.ipynb#quality-gates-and-quarantine)

*Keep loading good rows while bad ones wait for review.*

Using the same `batch`, build a `failures` array of rule names per row. Return `order_id` and `failures` (joined with `,`) for quarantined rows only, ordered by `order_id`.

In [ ]:
batch = orders.unionByName(spark.createDataFrame([(13, None, "paid", datetime(2026, 1, 14), D("10.00")), (14, 2, "lost", datetime(2026, 1, 14), D("-3.00"))], orders.schema))

**Expected output**

| order_id | failures |
| --- | --- |
| 13 | null_customer |
| 14 | negative_amount,bad_status |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
rules = {"null_customer": F.col("customer_id").isNull(), "negative_amount": F.col("amount") < 0, "bad_status": ~F.col("status").isin("placed", "paid", "shipped", "returned")}
checked = batch.withColumn("failures", F.array_compact(F.array(*[F.when(cond, F.lit(name)) for name, cond in rules.items()])))
result = checked.where(F.size("failures") > 0).select("order_id", F.array_join("failures", ",").alias("failures")).orderBy("order_id")
result.show()

## Exercise 7: Unit-test a transformation

**Difficulty:** Easy · about 10 min · **Practises:** [`assertDataFrameEqual` and `assertSchemaEqual`](pyspark-reference-15-production-patterns.ipynb#assertdataframeequal-and-assertschemaequal)

*Small tested functions are how production PySpark stays correct.*

Write `revenue_by_region(orders, customers)` (unknown region as `unknown`) and test it with `assert_df_equal` (schema plus rows, in any order) against `expected`. Return one row with `test` = `revenue_by_region` and `passed`.

In [ ]:
expected = spark.createDataFrame([("North", D("549.75")), ("South", D("459.00")), ("West", D("380.40")), ("unknown", D("1269.90"))], "region string, revenue decimal(22,2)")

**Expected output**

| test | passed |
| --- | --- |
| revenue_by_region | true |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def revenue_by_region(o, c):
    return o.join(c, "customer_id", "left").groupBy(F.coalesce("region", F.lit("unknown")).alias("region")).agg(F.sum("amount").alias("revenue"))

try:
    assert_df_equal(revenue_by_region(orders, customers), expected)
    passed = True
except AssertionError:
    passed = False
result = spark.createDataFrame([("revenue_by_region", passed)], "test string, passed boolean")
result.show()

## Exercise 8: Enforce a schema contract

**Difficulty:** Medium · about 10 min · **Practises:** [`assertDataFrameEqual` and `assertSchemaEqual`](pyspark-reference-15-production-patterns.ipynb#assertdataframeequal-and-assertschemaequal)

*Fail fast when an upstream schema changes.*

Check `feed_v1` and `feed_v2` against the contract `order_id int, status string, amount decimal(12,2)` with `assertSchemaEqual`. Return `feed` and `matches_contract`.

In [ ]:
contract = spark.createDataFrame([], "order_id int, status string, amount decimal(12,2)").schema
feed_v1 = orders.select("order_id", "status", "amount")
feed_v2 = orders.select("order_id", "status", F.col("amount").cast("double").alias("amount"))

**Expected output**

| feed | matches_contract |
| --- | --- |
| feed_v1 | true |
| feed_v2 | false |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
def matches(df):
    try:
        assertSchemaEqual(df.schema, contract)
        return True
    except AssertionError:
        return False

result = spark.createDataFrame([("feed_v1", matches(feed_v1)), ("feed_v2", matches(feed_v2))], "feed string, matches_contract boolean")
result.show()

## Exercise 9: Backfill a date range idempotently

**Difficulty:** Medium · about 15 min · **Practises:** [Write patterns and idempotency](pyspark-reference-15-production-patterns.ipynb#write-patterns-and-idempotency)

*Backfills replay many days through the same idempotent load.*

Create the Delta table at `bf` with revenue per day for all orders, then backfill 2026-01-05 to 2026-01-07 with amounts doubled (a corrected calculation) by replacing each day with `replaceWhere` in a loop. Return `day`, `revenue` ordered by `day`, for days up to 2026-01-09.

In [ ]:
bf = new_path("p15_backfill")
per_day = lambda df: df.groupBy(F.to_date("order_ts").alias("day")).agg(F.sum("amount").alias("revenue"))
per_day(orders).write.format("delta").partitionBy("day").save(bf)

**Expected output**

| day | revenue |
| --- | --- |
| 2026-01-05 | 469.00 |
| 2026-01-06 | 520.00 |
| 2026-01-07 | 408.50 |
| 2026-01-09 | 310.00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
corrected = orders.withColumn("amount", (F.col("amount") * 2).cast("decimal(12,2)"))
for d in ["2026-01-05", "2026-01-06", "2026-01-07"]:
    per_day(corrected).where(F.col("day") == F.lit(d).cast("date")).write.format("delta").mode("overwrite").option("replaceWhere", f"day = '{d}'").save(bf)
result = spark.read.format("delta").load(bf).where("day <= '2026-01-09'").orderBy("day")
result.show()

**Next:** look anything up in [PySpark reference 15: production patterns](pyspark-reference-15-production-patterns.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).